In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [2]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_gen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8,1.2]
)

# Training data
train_data = train_gen.flow_from_directory(
    "C:/Users/yasha/Documents/main_project/urban_issue_dataset",
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

# Validation data
val_data = train_gen.flow_from_directory(
    "C:/Users/yasha/Documents/main_project/urban_issue_dataset",
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

Found 3824 images belonging to 3 classes.
Found 955 images belonging to 3 classes.


In [3]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam

base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation='relu'),
    Dense(3, activation='softmax')
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 mobilenetv2_1.00_224 (Func  (None, 7, 7, 1280)        2257984   
 tional)                                                         
                                                                 
 global_average_pooling2d (  (None, 1280)              0         
 GlobalAveragePooling2D)                                         
                                                                 
 dense (Dense)               (None, 128)               163968    
                                                                 
 dense_1 (Dense)             (None, 3)                 387       
                                                                 
Total params: 2422339 (9.24 MB)
Trainable params: 164355 (642.01 KB)
Non-trainable params: 2257984 (8.61 MB)
_________________________________________________________________


In [4]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    callbacks=[early_stop]
)

Epoch 1/10
120/120 [==============================] - 1017s 8s/step - loss: 0.1155 - accuracy: 0.9582 - val_loss: 0.4728 - val_accuracy: 0.8471
Epoch 2/10
120/120 [==============================] - 220s 2s/step - loss: 0.0299 - accuracy: 0.9893 - val_loss: 0.5975 - val_accuracy: 0.8346
Epoch 3/10
120/120 [==============================] - 219s 2s/step - loss: 0.0180 - accuracy: 0.9932 - val_loss: 0.5522 - val_accuracy: 0.8649
Epoch 4/10
120/120 [==============================] - 218s 2s/step - loss: 0.0138 - accuracy: 0.9956 - val_loss: 0.4421 - val_accuracy: 0.8869
Epoch 5/10
120/120 [==============================] - 219s 2s/step - loss: 0.0105 - accuracy: 0.9961 - val_loss: 0.6515 - val_accuracy: 0.8534
Epoch 6/10
120/120 [==============================] - 219s 2s/step - loss: 0.0095 - accuracy: 0.9963 - val_loss: 0.6368 - val_accuracy: 0.8702
Epoch 7/10
120/120 [==============================] - 224s 2s/step - loss: 0.0085 - accuracy: 0.9963 - val_loss: 0.5718 - val_accuracy: 0.881

In [5]:



    model.save("model.keras")